# Task 1 — Data Exploration and Enrichment
Ethiopia Financial Inclusion Forecasting — Selam Analytics

Loads the unified dataset, validates it against `reference_codes.csv`, and summarizes what's in it before any modeling happens.

In [16]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
from data_loader import load_unified_data, load_reference_codes, validate_against_reference, split_by_record_type, get_indicator_series
df = load_unified_data()
ref = load_reference_codes()
df.head()

,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,unit,observation_date,period_start,period_end,fiscal_year,gender,location,region,source_name,source_type,source_url,confidence,related_indicator,parent_id,relationship_type,impact_direction,impact_magnitude,magnitude_estimate_pct,lag_months,evidence_basis,original_text,collected_by,collection_date,notes
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,%,2014-12-31,NaT,NaT,2014,all,national,NaN,Global Findex 2014,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,%,2017-12-31,NaT,NaT,2017,all,national,NaN,Global Findex 2017,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,%,2021-12-31,NaT,NaT,2021,all,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN
3,REC_0004,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,56.0,NaN,percentage,%,2021-12-31,NaT,NaT,2021,male,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN
4,REC_0005,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,36.0,NaN,percentage,%,2021-12-31,NaT,NaT,2021,female,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN


## Record counts by type

In [17]:
df["record_type"].value_counts()

record_type
observation    31
impact_link    16
event          10
target          3
Name: count, dtype: int64

## Schema validation
Every categorical field checked against `reference_codes.csv`.

In [18]:
issues = validate_against_reference(df, ref)
print("No issues found." if not issues else issues)

No issues found.


## Temporal range of observations

In [19]:
obs = df[df["record_type"]=="observation"]
print("Earliest:", obs["observation_date"].min().date())
print("Latest:  ", obs["observation_date"].max().date())

Earliest: 2014-12-31
Latest:   2025-12-31


## Indicators and their coverage

In [20]:
obs.groupby("indicator_code").agg(n_obs=("record_id","count"),
                                   first=("observation_date","min"),
                                   last=("observation_date","max"))

,n_obs,first,last
indicator_code,,,
ACC_4G_COV,2,2023-06-30,2025-06-30
ACC_FAYDA,3,2024-08-15,2025-05-15
ACC_MM_ACCOUNT,2,2021-12-31,2024-11-29
ACC_MOBILE_PEN,1,2025-12-31,2025-12-31
ACC_OWNERSHIP,6,2014-12-31,2024-11-29
AFF_DATA_INCOME,1,2024-12-31,2024-12-31
GEN_GAP_ACC,2,2021-12-31,2024-11-29
GEN_GAP_MOBILE,1,2024-12-31,2024-12-31
GEN_MM_SHARE,1,2024-12-31,2024-12-31


## Cataloged events

In [21]:
df[df["record_type"]=="event"][["record_id","category","indicator","observation_date"]]\
  .sort_values("observation_date")

,record_id,category,indicator,observation_date
33,EVT_0001,product_launch,Telebirr Launch,2021-05-17
41,EVT_0009,policy,NFIS-II Strategy Launch,2021-09-01
34,EVT_0002,market_entry,Safaricom Ethiopia Commercial Launch,2022-08-01
35,EVT_0003,product_launch,M-Pesa Ethiopia Launch,2023-08-01
36,EVT_0004,infrastructure,Fayda Digital ID Program Rollout,2024-01-01
37,EVT_0005,policy,Foreign Exchange Liberalization,2024-07-29
38,EVT_0006,milestone,P2P Transaction Count Surpasses ATM,2024-10-01
39,EVT_0007,partnership,M-Pesa EthSwitch Integration,2025-10-27
42,EVT_0010,pricing,Safaricom Ethiopia Price Increase,2025-12-15
40,EVT_0008,infrastructure,EthioPay Instant Payment System Launch,2025-12-18


## Impact links — event to indicator relationships
**Note on enrichment**: the starter dataset shipped with 0 `impact_link` records despite events and indicators both being present. All 16 rows below were added during enrichment — see `../data_enrichment_log.md` and `../reports/impact_links_methodology.md` for full sourcing and reasoning.

In [22]:
links = df[df["record_type"]=="impact_link"]
links[["record_id","parent_id","pillar","related_indicator","relationship_type",
       "impact_direction","impact_magnitude","evidence_basis","confidence"]]

,record_id,parent_id,pillar,related_indicator,relationship_type,impact_direction,impact_magnitude,evidence_basis,confidence
44,IMP_0001,EVT_0001,ACCESS,ACC_MM_ACCOUNT,direct,increase,high,empirical,estimated
45,IMP_0002,EVT_0003,ACCESS,ACC_MM_ACCOUNT,direct,increase,medium,empirical,estimated
46,IMP_0003,EVT_0001,ACCESS,ACC_OWNERSHIP,indirect,increase,low,empirical,estimated
47,IMP_0004,EVT_0003,ACCESS,ACC_OWNERSHIP,indirect,increase,negligible,empirical,estimated
48,IMP_0005,EVT_0001,USAGE,USG_DIGITAL_PAYMENT,direct,increase,high,empirical,estimated
49,IMP_0006,EVT_0003,USAGE,USG_DIGITAL_PAYMENT,direct,increase,medium,empirical,estimated
50,IMP_0007,EVT_0004,ACCESS,ACC_OWNERSHIP,enabling,increase,low,literature,low
51,IMP_0008,EVT_0004,USAGE,USG_DIGITAL_PAYMENT,enabling,increase,low,literature,low
52,IMP_0009,EVT_0005,USAGE,USG_DIGITAL_PAYMENT,indirect,increase,low,theoretical,low
53,IMP_0010,EVT_0002,ACCESS,ACC_MOBILE_PEN,direct,increase,low,theoretical,low


## Enrichment additions this round
- `REC_0034`: Digital Payment Adoption Rate (USG_DIGITAL_PAYMENT), 35%, 2024 — the Usage forecast target had **zero** observations before this addition.
- 16 `impact_link` records connecting all 10 events to specific Access/Usage indicators (see Task 3 notebook for the association matrix built from these).

Full documentation: `../data_enrichment_log.md`.